# ChuckleNet: WavLM-Base+ Embedding Extraction

Extract 768-dim WavLM embeddings for **15,000 balanced utterances** from stand-up comedy.

**Dataset**: 15K balanced utterances (32% positive) from 71 videos
**Model**: `microsoft/wavlm-base-plus` → 768-dim embeddings
**Output**: `wavlm_utterance_embeddings_15k.pt` (768 × 15,000)
**Runtime**: ~1.5-2 hours on T4 GPU

**Context**:
- Phase 1 text-only F1 = 0.44 (utterance-level)
- Phase 2 fusion target: F1 = 0.55-0.65
- Phase 3 joint target: F1 > 0.85

**After running**: Copy `wavlm_utterance_embeddings_15k.pt` to:
`data/audio_comedy/wavlm_utterance_embeddings.pt`

In [ ]:
#@title 1. Setup
!pip install transformers torch torchaudio datasets soundfile -q
!nvidia-smi

In [ ]:
#@title 2. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
BASE_DIR = '/content/drive/MyDrive/laughter_prediction_backup'
assert os.path.exists(BASE_DIR), f'GDrive backup not found at {BASE_DIR}'
print(f'✅ GDrive mounted: {BASE_DIR}')

In [ ]:
#@title 3. Check available data
import json, glob

UTTERANCES_PATH = f'{BASE_DIR}/aligned_utterances_balanced15k.jsonl'
AUDIO_DIR = f'{BASE_DIR}/audio'

# Try the 15K balanced file, fall back to aligned_utterances.jsonl
if not os.path.exists(UTTERANCES_PATH):
    UTTERANCES_PATH = f'{BASE_DIR}/aligned_utterances.jsonl'

assert os.path.exists(UTTERANCES_PATH), f'Utterances not found at {UTTERANCES_PATH}'

utterances = []
with open(UTTERANCES_PATH) as f:
    for line in f:
        utterances.append(json.loads(line))

print(f'Loaded {len(utterances)} utterances')
pos = sum(1 for u in utterances if u['label_any'] == 1)
print(f'Positive: {pos} ({100*pos/len(utterances):.1f}%)')
print(f'Negative: {len(utterances)-pos} ({100*(len(utterances)-pos)/len(utterances):.1f}%)')

# Find audio files
audio_map = {}
for ext in ['*.mp3', '*.wav', '*.m4a', '*.opus']:
    for path in glob.glob(f'{AUDIO_DIR}/**/{ext}', recursive=True):
        vid = os.path.basename(path).rsplit('.', 1)[0]
        audio_map[vid] = path

covered = sum(1 for u in utterances if u['video_id'] in audio_map)
print(f'Audio files found: {len(audio_map)}')
print(f'Utterances with audio: {covered}/{len(utterances)} ({100*covered/len(utterances):.1f}%)')

In [ ]:
#@title 4. Load WavLM-Base+
import torch
from transformers import WavLMModel

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

wavlm = WavLMModel.from_pretrained('microsoft/wavlm-base-plus')
wavlm = wavlm.to(device)
wavlm.eval()
print(f'✅ WavLM-Base+ loaded ({sum(p.numel() for p in wavlm.parameters()):,} params)')

In [ ]:
#@title 5. Extract WavLM Embeddings
import torchaudio
from tqdm.auto import tqdm

SAMPLE_RATE = 16000
CONTEXT_PAD = 0.2  # 200ms padding

embeddings = {}
errors = []
audio_cache = {}  # LRU cache

def load_audio(video_id):
    """Load audio for a video (cached)."""
    if video_id in audio_cache:
        return audio_cache[video_id]
    path = audio_map.get(video_id)
    if path is None:
        return None, None
    try:
        wav, sr = torchaudio.load(path)
        if sr != SAMPLE_RATE:
            wav = torchaudio.functional.resample(wav, sr, SAMPLE_RATE)
        if wav.shape[0] > 1:
            wav = wav.mean(dim=0, keepdim=True)
        wav = wav.squeeze(0)
        audio_cache[video_id] = wav
        return wav, SAMPLE_RATE
    except Exception as e:
        errors.append((video_id, str(e)))
        return None, None

for i, utt in enumerate(tqdm(utterances, desc='Extracting')):
    uid = utt['utterance_id']
    vid = utt['video_id']
    
    wav, sr = load_audio(vid)
    if wav is None:
        continue
    
    # Extract clip with context padding
    start_s = max(0, int((utt['start'] - CONTEXT_PAD) * SAMPLE_RATE))
    end_s = min(len(wav), int((utt['end'] + CONTEXT_PAD) * SAMPLE_RATE))
    clip = wav[start_s:end_s]
    
    if len(clip) < SAMPLE_RATE * 0.1:
        continue
    
    # Forward pass
    with torch.no_grad():
        inp = clip.unsqueeze(0).to(device)
        out = wavlm(inp)
        emb = out.last_hidden_state.mean(dim=1).squeeze(0).cpu()
    
    embeddings[uid] = emb
    
    # Clear cache every 1000 utterances
    if i > 0 and i % 1000 == 0:
        recent_vids = {u['video_id'] for u in utterances[max(0,i-500):i+500]}
        audio_cache = {k: v for k, v in audio_cache.items() if k in recent_vids}
        torch.cuda.empty_cache()

print(f'\n✅ Extracted {len(embeddings)} embeddings')
print(f'❌ Errors: {len(errors)}')
if errors:
    for vid, err in errors[:5]:
        print(f'  {vid}: {err}')

In [ ]:
#@title 6. Save embeddings
OUTPUT_PATH = f'{BASE_DIR}/wavlm_utterance_embeddings_15k.pt'

embedding_ids = sorted(embeddings.keys())
embedding_tensor = torch.stack([embeddings[uid] for uid in embedding_ids])

print(f'Embedding tensor: {embedding_tensor.shape}')
print(f'Expected: ({len(embedding_ids)}, 768)')

save_dict = {
    'embeddings': embedding_tensor,
    'utterance_ids': embedding_ids,
    'model': 'microsoft/wavlm-base-plus',
    'pooling': 'mean',
    'context_padding': CONTEXT_PAD,
    'n_utterances': len(embedding_ids),
    'n_positive': sum(1 for uid in embedding_ids if any(u['utterance_id']==uid and u['label_any']==1 for u in utterances)),
}

torch.save(save_dict, OUTPUT_PATH)
print(f'\n✅ Saved to {OUTPUT_PATH}')
print(f'   Size: {os.path.getsize(OUTPUT_PATH) / 1024**2:.1f} MB')

In [ ]:
#@title 7. Verify
data = torch.load(OUTPUT_PATH)
print(f'Shape: {data["embeddings"].shape}')
print(f'Model: {data["model"]}')
print(f'N utterances: {data["n_utterances"]}')
print(f'Pooling: {data["pooling"]}')
print(f'\n✅ Ready for Phase 2 fusion training!')

# Copy to expected local path
LOCAL_PATH = '/content/drive/MyDrive/laughter_prediction/audio_comedy/wavlm_utterance_embeddings.pt'
os.makedirs(os.path.dirname(LOCAL_PATH), exist_ok=True)
torch.save(save_dict, LOCAL_PATH)
print(f'Also saved to: {LOCAL_PATH}')